# 07 — REDCap API Demo
**NANO Study | ESD Lab, University of South Carolina**

> ⚠️ **HIPAA Notice**: Uses **masked/mock credentials** only. Never commit real API tokens. Real credentials go in `.env` (gitignored).

## Goals
1. Demonstrate REDCap API pull using `redcap` Python library with mocked responses
2. Show record export by event with field filtering
3. Demonstrate completeness audit report generation
4. Show how to push validated data back to REDCap

**Production usage**: See `redcap/api/redcap_pull.py` and `scripts/redcap_daily_sync.py`

In [ ]:
import sys
import json
import warnings
from pathlib import Path
from unittest.mock import MagicMock, patch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

rng = np.random.default_rng(seed=99)
print('Setup complete. Using MOCK REDCap credentials for demo.')

## 1. Configuration Loading (via config/redcap_config.yml)

In production, credentials come exclusively from environment variables. Here we use mock values.

In [ ]:
# MOCK configuration — never use real values here
MOCK_CONFIG = {
    'api_url':    'https://redcap.your-institution.edu/api/',
    'api_token':  'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX',  # Masked
    'project_id': 12345,
    'events': [
        'nicu_admission_arm_1', 'month_1_arm_1', 'month_2_arm_1',
        'month_3_arm_1', 'month_6_arm_1', 'month_9_arm_1',
        'month_12_arm_1', 'month_36_arm_1'
    ],
    'study_arms': {
        'ASIB': 'arm_1', 'PT': 'arm_2', 'TD': 'arm_3'
    }
}

print('REDCap configuration loaded (MOCK):')
print(f"  API URL: {MOCK_CONFIG['api_url']}")
print(f"  Token:   {'*' * 32} (masked)")
print(f"  Events:  {len(MOCK_CONFIG['events'])} longitudinal events")
print(f"  Arms:    {list(MOCK_CONFIG['study_arms'].keys())}")

## 2. Mock REDCap API Pull

Simulates the output of `PyCap.Project.export_records()` with realistic NANO study data.

In [ ]:
def generate_mock_redcap_records(n_participants: int = 30) -> list:
    """Generate mock REDCap export records for NANO study demo."""
    records = []
    groups = rng.choice(['ASIB', 'PT', 'TD'], n_participants, p=[0.2, 0.5, 0.3])
    
    for pid in range(1, n_participants + 1):
        group = groups[pid - 1]
        nano_id = f'NANO-{pid:04d}'
        ga = round(float(rng.normal(29, 2)), 1)
        
        for event_idx, event in enumerate(MOCK_CONFIG['events']):
            if rng.random() < 0.08 * event_idx:  # increasing missingness
                continue
            records.append({
                'record_id':            nano_id,
                'nano_id':              nano_id,
                'redcap_event_name':    event,
                'group':                group,
                'ga_weeks':             ga,
                'sex':                  rng.choice(['1', '2']),
                'dob':                  '1970-01-01',  # Placeholder (not real DOB)
                'rsa':                  round(float(rng.normal(4.1, 0.8)), 3),
                'rmssd_ms':             round(float(rng.normal(34, 10)), 2),
                'ecg_recording_complete': str(rng.choice([0, 2], p=[0.1, 0.9])),
                'ados_css':             str(rng.integers(1, 20)) if (event == 'month_36_arm_1' and group == 'ASIB') else '',
                'bayley_cognitive':     str(round(float(rng.normal(88, 15)))) if event in ['month_12_arm_1', 'month_36_arm_1'] else '',
                'redcap_repeat_instance': '',
            })
    return records

mock_records = generate_mock_redcap_records(n_participants=30)
df_rc = pd.DataFrame(mock_records)
print(f'Mock export: {len(mock_records)} records, {df_rc["record_id"].nunique()} participants')
df_rc.head(5)

## 3. Completeness Audit Report

In [ ]:
# Completeness matrix: participant x event
all_events = MOCK_CONFIG['events']
all_ids = df_rc['record_id'].unique()

completeness = pd.DataFrame(index=all_ids, columns=all_events, data=0)
for _, row in df_rc.iterrows():
    if row['redcap_event_name'] in completeness.columns:
        completeness.loc[row['record_id'], row['redcap_event_name']] = 1

completeness = completeness.astype(int)

# Completion rate per event
event_pct = completeness.mean() * 100
print('Completion rate per event:')
for ev, pct in event_pct.items():
    status = '✓' if pct >= 80 else '⚠' if pct >= 60 else '✗'
    print(f'  {status} {ev:40s}: {pct:.1f}%')

# Heatmap
fig, ax = plt.subplots(figsize=(12, 8))
shortened_events = [e.replace('_arm_1', '').replace('_', ' ') for e in all_events]
sns.heatmap(completeness.values, ax=ax,
            cmap='RdYlGn', vmin=0, vmax=1, cbar=False,
            yticklabels=False, xticklabels=shortened_events,
            linewidths=0.01, linecolor='lightgray')
ax.set_xlabel('Visit Event')
ax.set_ylabel('Participants')
ax.set_title('REDCap Completeness Heatmap (Mock Data)\nGreen = Complete, Red = Missing')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Mock REDCap Push (Validation Demo)

In production, `redcap/api/redcap_push.py` validates data before pushing. Here we demonstrate the validation logic.

In [ ]:
def validate_records_for_push(df: pd.DataFrame) -> dict:
    """Validate records before pushing to REDCap API."""
    errors = []
    warnings_list = []

    # 1. Check NANO ID format
    import re
    bad_ids = df[~df['nano_id'].str.match(r'^NANO-\d{4}$')]['nano_id'].unique()
    if len(bad_ids):
        errors.append(f'{len(bad_ids)} records with invalid NANO ID format')

    # 2. Check physiological ranges
    if 'rsa' in df.columns:
        out_of_range = df[(df['rsa'].notna()) & ((df['rsa'] < 0) | (df['rsa'] > 12))]
        if len(out_of_range):
            errors.append(f'{len(out_of_range)} records with RSA outside [0, 12]')

    if 'rmssd_ms' in df.columns:
        extreme = df[(df['rmssd_ms'].notna()) & ((df['rmssd_ms'] < 1) | (df['rmssd_ms'] > 200))]
        if len(extreme):
            warnings_list.append(f'{len(extreme)} records with RMSSD outside expected range [1, 200 ms]')

    return {
        'valid': len(errors) == 0,
        'errors': errors,
        'warnings': warnings_list,
        'records_checked': len(df),
    }

# Validate mock data
validation = validate_records_for_push(df_rc)
print(f'Validation result: {"PASS" if validation["valid"] else "FAIL"}')
print(f'Records checked: {validation["records_checked"]}')
print(f'Errors: {validation["errors"] or "None"}')
print(f'Warnings: {validation["warnings"] or "None"}')

if validation['valid']:
    print('\n✓ Data would be pushed to REDCap (simulation only — no real API call made)')
else:
    print('\n✗ Push BLOCKED — fix validation errors before pushing')

## Summary

This demo shows:
- How to load REDCap config from YAML (with masked token)
- Mock pull structure matching `redcap/api/redcap_pull.py` output format
- Completeness audit: per-participant × per-event heatmap
- Pre-push validation: NANO ID format, physiological range checks

**Production workflow**:
```bash
# Set credentials in .env (gitignored)
export REDCAP_API_TOKEN=your_token_here
export NANO_DATA_ROOT=/path/to/secure/drive

# Run daily sync
python scripts/redcap_daily_sync.py
```

See `redcap/api/redcap_pull.py` and `redcap/api/redcap_audit.py` for production implementations.